[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/10_gqa.ipynb)

# 🔴 Hard: Grouped Query Attention (GQA)

Implement **Grouped Query Attention** — used in LLaMA 2, Mistral, etc. to reduce KV cache size.

Like MHA, but with **fewer KV heads** than Q heads. Each group of Q heads shares the same K/V head.

### Signature
```python
class GroupQueryAttention:
    def __init__(self, d_model: int, num_heads: int, num_kv_heads: int): ...
    def forward(self, x) -> torch.Tensor:  # self-attention
```

### Requirements
- `self.W_q`: `nn.Linear(d_model, d_model)` — full Q projection
- `self.W_k`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced K projection
- `self.W_v`: `nn.Linear(d_model, num_kv_heads * d_k)` — reduced V projection
- `self.W_o`: `nn.Linear(d_model, d_model)` — output projection
- `d_k = d_model // num_heads`
- Expand KV heads with `repeat_interleave` to match Q heads
- When `num_kv_heads == num_heads`, should behave like standard MHA

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 3.3 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
def sequence_mask(X, valid_len, value=0):
    """Mask irrelevant entries in sequences.

    Defined in :numref:`sec_utils`"""
    maxlen = X.size(1)
    mask = torch.arange((maxlen), dtype=torch.float32,
                        device=X.device)[None, :] < valid_len[:, None]
    X[~mask] = value
    return X

def masked_softmax(X,valid_lens):
    "通过在最后一个轴上遮蔽元素来执行softmax操作"
    if valid_lens is None:
        return nn.functional.softmax(X,dim=-1)
    else:
        shape = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens,shape[1])
        #展平成一维的，[1,2,3,4,1,2,3,4………………] 总共有batch*num_heads*num_step个数字
        else:
            valid_lens = valid_lens.reshape(-1)
        #X.reshape(-1,shape[-1]),X：batch*num_head*num_query, num_key_value 每一行表示一个query对应的分数
        X = sequence_mask(X.reshape(-1,shape[-1]),valid_lens,value=-1e6)
        return nn.functional.softmax(X.reshape(shape),dim=-1)

In [ ]:
class DotProductAttention(nn.Module):
    def __init__(self,dropout,**kwargs):
        super(DotProductAttention,self).__init__(**kwargs)
        self.dropout = nn.Dropout(dropout)
    def forward(self,queries,keys,values,valid_lens=None):
        d = queries.shape[-1]
        scores = torch.bmm(queries,keys.transpose(1,2))  / math.sqrt(d)
        self.attention_weights = masked_softmax(scores,valid_lens)
        return torch.bmm(self.dropout(self.attention_weights),values)

In [ ]:
# # ✏️ YOUR IMPLEMENTATION HERE

# class GroupQueryAttention:
#     def __init__(self, d_model, num_heads, num_kv_heads):
#         pass  # Initialize projections

#     def forward(self, x):
#         pass  # Self-attention with grouped KV

In [ ]:
class GroupQueryAttention(nn.Module):
    """分组查询注意力 GQA"""
    def __init__(self, d_model, num_heads, num_kv_heads,dropout=0,bias=False):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads        # Query 头总数
        self.num_kv_heads = num_kv_heads  # KV 头总数
        self.d_k = d_model // num_heads   # 单个头的维度

        # 线性投影层
        self.W_q = nn.Linear(d_model, d_model, bias=bias)
        #k,v需要投影为 num_kv_heads * self.d_k（因为头数减少了）
        self.W_k = nn.Linear(d_model, num_kv_heads * self.d_k, bias=bias)
        self.W_v = nn.Linear(d_model, num_kv_heads * self.d_k, bias=bias)
        self.W_o = nn.Linear(d_model, d_model, bias=bias)

        # 点积注意力模块
        self.attention = DotProductAttention(dropout)

    def forward(self, x, valid_lens=None):
        # 自注意力：输入 x 同时作为 queries / keys / values
        batch_size, seq_len, _ = x.shape

        # 由单个输入 x 投影得到 Q、K、V
        q = self.W_q(x)
        k = self.W_k(x)
        v = self.W_v(x)

        # Q 做多头变换（标准多头注意力机制）
        q = transpose_qkv(q, self.num_heads)  # (batch*num_heads, seq_q, d_k)

        # K、V 变换为 KV 头格式
        k = k.reshape(batch_size, seq_len, self.num_kv_heads, self.d_k)
        k = k.permute(0, 2, 1, 3)

        v = v.reshape(batch_size, seq_len, self.num_kv_heads, self.d_k)
        v = v.permute(0, 2, 1, 3)

        # 扩充 KV 头，匹配 Query 头数量
        # 计算每组 KV 对应多少个 Q 头
        repeat_times = self.num_heads // self.num_kv_heads
        # 维度变换 + 重复扩充
        k = k.reshape(batch_size * self.num_kv_heads, seq_len, self.d_k)
        k = torch.repeat_interleave(k, repeats=repeat_times, dim=0)

        v = v.reshape(batch_size * self.num_kv_heads, seq_len, self.d_k)
        v = torch.repeat_interleave(v, repeats=repeat_times, dim=0)

        # 处理 valid_lens 掩码
        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(valid_lens, repeats=self.num_heads, dim=0)

        # 缩放点积注意力计算
        attn_out = self.attention(q, k, v, valid_lens)
        attn_weights = self.attention.attention_weights

        # 拼接多头输出 + 最终投影
        output_concat = transpose_output(attn_out, self.num_heads)
        final_out = self.W_o(output_concat)

        return final_out

In [ ]:
def transpose_qkv(X, num_heads):
    """为了多注意力头的并行计算而变换形状"""
    # 输入X的形状:(batch_size，查询或者“键－值”对的个数，num_hiddens)
    # 输出X的形状:(batch_size，查询或者“键－值”对的个数，num_heads，
    # num_hiddens/num_heads)
    X = X.reshape(X.shape[0], X.shape[1], num_heads, -1)

    # 输出X的形状:(batch_size，num_heads，查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    X = X.permute(0, 2, 1, 3)

    # 最终输出的形状:(batch_size*num_heads,查询或者“键－值”对的个数,
    # num_hiddens/num_heads)
    return X.reshape(-1, X.shape[2], X.shape[3])


def transpose_output(X, num_heads):
    """逆转transpose_qkv函数的操作"""
    X = X.reshape(-1, num_heads, X.shape[1], X.shape[2])
    X = X.permute(0, 2, 1, 3)
    #还原回batch,num_query,向量维度
    return X.reshape(X.shape[0], X.shape[1], -1)

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
gqa = GroupQueryAttention(d_model=32, num_heads=8, num_kv_heads=2, dropout=0)
print("W_q shape:", gqa.W_q.weight.shape)  # (32, 32)
print("W_k shape:", gqa.W_k.weight.shape)  # (8, 32)  — only 2 KV heads * d_k=4

x = torch.randn(2, 6, 32)
out = gqa.forward(x)
print("Output shape:", out.shape)           # (2, 6, 32)

W_q shape: torch.Size([32, 32])
W_k shape: torch.Size([8, 32])
Output shape: torch.Size([2, 6, 32])


In [ ]:
from torch_judge import check
check('gqa')


🧪 Testing: Grouped Query Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/5] Output shape (5.7ms)
  ✅ [2/5] nn.Linear with correct shapes (2.8ms)
  ✅ [3/5] Degenerates to MHA when kv_heads == heads (5.8ms)
  ✅ [4/5] KV heads are shared correctly (4.1ms)
  ✅ [5/5] Gradient flow (46.8ms)
──────────────────────────────────────────────────
  🎉 All 5 tests passed! (65.3ms total)
  Progress saved. Run status() to see your dashboard.

